# NPTEL Study Notes: Introduction to Large Language Models (LLMs)

## 1. Historical Context & Core Intuition

The foundational concept of predicting language dates back to the 1950s with Claude Shannon’s **character guessing game**.

* **The Game:** Given a sequence of characters, guess the next one. On average, a human can guess the correct English character in 3 to 4 attempts.
* **Shannon's Entropy:** This 3 to 4 attempt average closely aligns with the entropy of the English language (approximately 4.2 to 4.3).
* **Perplexity Connection:** Entropy measures the randomness or average information. In language modeling, this is tied to **perplexity**, the standard metric used to evaluate how well a model predicts a sequence (to be covered in future lectures).

*(Note: The transcript mentions "sinus entropy", which is a transcription error for "Shannon's entropy".)*

## 2. What is a Language Model?

A Statistical Language Model is fundamentally a **probability distribution over a sequence of tokens** (which can be characters, words, or subwords). Its job is to determine how likely a given sequence of text is to occur.

### Key Applications

* **Speech Recognition:** Resolving phonetic ambiguities. If a user says something that sounds like "ma or kit", the model assigns a higher probability to "market".
* **Machine Translation:** Capturing context and semantic nuance. The model knows that "heavy rainfall" is far more probable than "big rainfall," and "festival of lights" is more probable than "festival of lamps."
* **Search Auto-completion:** Predicting the next most likely word in a search query.

> **Key Insight:** Language models do not just memorize syntax; they effectively capture world knowledge and contextual semantics through probabilistic word associations.

---

## 3. Mathematical Foundation & N-gram Models

To find the probability of a sequence of $n$ tokens $W_1, W_2, \dots, W_n$, we use the **Chain Rule of Probability**:

$$P(W_1, W_2, \dots, W_n) = P(W_1) \times P(W_2\vert{}W_1) \times P(W_3\vert{}W_1, W_2) \dots P(W_n\vert{}W_1, \dots, W_{n-1})$$

### The Sparsity Problem

To calculate the conditional probability using Maximum Likelihood Estimation (MLE), we divide raw counts from our corpus. For example:

$$P(\text{season} \vert{} \text{the monsoon}) = \frac{\text{Count}(\text{the monsoon season})}{\text{Count}(\text{the monsoon})}$$

**The Flaw:** As the context length increases, the chance of finding that exact phrase in the training corpus drops to zero. If even one conditional probability is zero, the probability of the entire sequence becomes zero.

### The Markov Assumption

To fix the sparsity problem, we apply a **Markov Assumption**, which states that the probability of a word only depends on a limited history of previous words, rather than the entire context.

| Model Type | Markov Order | Context Considered | Formula |
| --- | --- | --- | --- |
| **Unigram** | 0th Order | None (Words are independent) | $P(W_n)$ |
| **Bigram** | 1st Order | Previous 1 word | $P(W_n \vert{} W_{n-1})$ |
| **Trigram** | 2nd Order | Previous 2 words | $P(W_n \vert{} W_{n-1}, W_{n-2})$ |
| **N-gram** | $(N-1)$th Order | Previous $N-1$ words | $P(W_n \vert{} W_{n-N+1}, \dots, W_{n-1})$ |

**The Trade-off:**

* **Lower-order models (e.g., Unigram):** No zero probabilities, but the generated text lacks grammar and meaning.
* **Higher-order models (e.g., 5-gram):** Better grammar, but severe data sparsity (many zero probabilities).
* **Sweet Spot:** 3-gram or 4-gram models typically offer the best balance.

### Limitation: Long-Range Dependencies

N-gram models fail when grammar depends on words far back in the sentence.

* *Example:* "The **project** which he had been working on for months was finally **approved**."
* A trigram model looking only at "months was finally" cannot logically predict "approved" because the subject ("project") is outside its context window. This limitation is later solved by Recurrent Neural Networks (RNNs/LSTMs).

---

## 4. The Bigram Count Matrix

A bigram model can be represented as a square matrix where rows are the first word ($W_1$) and columns are the second word ($W_2$).

**Mathematical Property:** Assuming you properly include Start-of-Sentence and End-of-Sentence tokens, the sum of a specific word's row entries will equal the sum of its column entries. This is because every time a word appears in the corpus, it acts as the second word of a bigram exactly as many times as it acts as the first word of a following bigram.

**Sparsity & Contingent Zeros:**
In large vocabularies, **99.95%** of the matrix entries are zero.

* Some zeros just represent phrases missing from the training data (e.g., "enjoyed the festival").
* **Contingent Zeros** represent grammatically possible but highly unnatural phrases that truly rarely occur (e.g., "to food" as in "I went to food stall").

---

## 5. Handling Unknowns: Out-of-Vocabulary (OOV) Tokens

A test sentence may contain words that never appeared in the training set (e.g., modern slang like "LOL"). If a word has no row/column in our matrix, the model breaks.

**The Solution:**

1. Scan the training corpus and count the frequency of all tokens.
2. Establish a frequency threshold (e.g., 5).
3. Move all tokens above the threshold into a **Lexicon** (your actual working vocabulary).
4. Replace all tokens below the threshold in the training data with a special `<UNK>` (Unknown) token.
5. During testing, any word not found in the Lexicon is treated as `<UNK>`, and its probability is calculated using the `<UNK>` statistics gathered during training.

---

## 6. Smoothing Techniques

Smoothing solves the problem of valid phrases receiving a zero probability simply because they didn't appear in the training corpus.

### A. Add-1 (Laplace) Smoothing

Blindly add 1 to every bigram count. To ensure probabilities still sum to 1, we must add the vocabulary size, $\vert{}V\vert{}$, to the denominator.

$$P_{Add-1}(W_i \vert{} W_{i-1}) = \frac{C(W_{i-1}, W_i) + 1}{C(W_{i-1}) + \vert{}V\vert{}}$$

* **Drawback:** It "steals from the rich and gives to the poor." It shifts too much probability mass from high-frequency words to zero-frequency words, drastically distorting actual data distributions.

### B. Add-k Smoothing

Instead of adding 1, add a fractional value $k$ to the numerator and $k\vert{}V\vert{}$ to the denominator.

$$P_{Add-k}(W_i \vert{} W_{i-1}) = \frac{C(W_{i-1}, W_i) + k}{C(W_{i-1}) + k\vert{}V\vert{}}$$

* $k$ acts as a hyperparameter tuned on a held-out validation dataset.
* This can be further optimized by replacing the uniform addition with an addition proportional to the word's unigram probability.

### C. Backoff & Interpolation (Advanced)

* **Backoff:** If a trigram probability is zero, "back off" and check the bigram probability. If the bigram is zero, back off to the unigram probability.
* **Interpolation:** Always calculate a linear combination of all n-gram probabilities.

$$P_{interp} = \lambda_1 P_{trigram} + \lambda_2 P_{bigram} + \lambda_3 P_{unigram}$$


*(Where $\lambda_1 + \lambda_2 + \lambda_3 = 1$. These lambdas can be static constraints or dynamic functions based on context).*

*(Note: The transcript ends by mentioning "Naive smoothing" for the next lecture. This is almost certainly a transcription error for **Kneser-Ney smoothing**, which is traditionally taught alongside Good-Turing in this exact context).*

---

## 7. Practice Questions

**Q1. A trigram language model represents which order of Markov assumption?**
A) 1st order
B) 2nd order
C) 3rd order
D) 4th order

**Answer:** B) 2nd order.
*Explanation:* An N-gram model relies on an $(N-1)$th order Markov assumption because it looks at the previous $N-1$ states (words) to predict the current one. A trigram ($N=3$) looks at the 2 previous words.

**Q2. In a training corpus, the word "artificial" appears 40 times. The phrase "artificial intelligence" appears 0 times. The total vocabulary size $\vert{}V\vert{}$ is 10,000. Using Add-1 (Laplace) smoothing, what is the probability $P(\text{intelligence} \vert{} \text{artificial})$?**

**Answer:**
Let's define the terms based on the Add-1 smoothing formula:
$C(W_{i-1}, W_i) = C(\text{artificial, intelligence}) = 0$
$C(W_{i-1}) = C(\text{artificial}) = 40$
$\vert{}V\vert{} = 10,000$

$$P_{Add-1} = \frac{0 + 1}{40 + 10,000} = \frac{1}{10,040}$$

**Q3. Why is the row sum equivalent to the column sum for a specific word in a complete Bigram Count Matrix (assuming sentence boundaries are tokenized)?**

**Answer:** Because every word in the interior of a corpus acts as the first word of a bigram exactly the same number of times it acts as the second word of a bigram. If we tokenize the start and end of a document/sentence properly, every entrance into a word token is matched by a subsequent exit to the next token, ensuring perfect mathematical symmetry in total row/column counts.

---

**Revision Summary:** Statistical language models assign probabilities to token sequences. To bypass the sparsity of exact phrase matches, we use the Markov assumption to restrict context (N-grams). While higher N-grams capture better grammar, they suffer from data sparsity. We handle unseen test words by mapping low-frequency training words to an `<UNK>` token, and we solve zero-probabilities for unseen word combinations using smoothing techniques like Add-1, Add-k, Backoff, and Interpolation.